In [1]:
# this notebook is perform anaylsis of MACE events in the care home hypertension cohort 

In [2]:
library(ggplot2)
library(bigrquery)
library(tidyverse)
library(ggplot2)
library(lubridate)
library(stats)

ProjectId = "yhcr-prd-bradfor-bia-core"

targetdb1 <-'yhcr-prd-bradfor-bia-core.CB_MYSPACE_MK'
targetdb1 <-gsub(' ','',targetdb1)

targetdb2 <- 'yhcr-prd-bradfor-bia-core.CB_FDM_DeathCertificates'
targetdb2 <-gsub(' ','',targetdb2)

targetdb3 <- 'yhcr-prd-bradfor-bia-core.CB_2172'
targetdb3 <- gsub(' ','',targetdb3)

In [33]:
sql1 <- 
paste(
    'SELECT  *
from ',targetdb3,'.cb_2172_mace 
where person_id in(select person_id from ',targetdb3,'.care_home_cohort_v1) 
;',sep="")

mace_tbl <-bq_project_query(ProjectId,sql1)
mace_data <- bq_table_download(mace_tbl)
#filter out any individuals where the time between admission and death is less than 42 days 


In [34]:
mace_data$date.event <- as.Date(mace_data$dod)

mace_data <- mace_data %>% select(-dod)

In [35]:
mace_data <- mace_data %>% arrange(person_id, desc(date.event)) %>% mutate(event_id= rank(date.event
                                                                                        ))
#here is the initial data, the problem is there are some  duplicate events where the data differs by only a couple of days so they are counted as different events. 
#Some cardiovasccular deaths also have  the cardiovascular event (e.g. myocardial evvent,
#recorded as a seperate event a few days apart). I need to remove these events so that only the cardiovascular death is recorded. 
#This is based on the assumption that where a stroke or MI has been recorded within 30 days of a cardiovascular death, 
# the two are realted and therefore the stroke or MI was a fatal one so should be recorded as a CV death 
#rather than a non fatal MI or stroke. 

person_id,event_type,date.event,event_id
<int>,<chr>,<date>,<dbl>
339912,cv death,2017-12-15,70.5
533373,cv death,2017-11-15,63.5
576388,cv death,2019-10-15,170.0
706765,cv death,2019-03-15,133.5
879366,cv death,2019-06-15,154.0
879366,myocardial infarction,2019-06-10,151.0
899131,myocardial infarction,2020-05-26,191.0
899131,cv death,2020-05-15,190.0
12388259,cv death,2017-02-15,16.5


In [36]:
# this identifies myocardial infarction events where another event occurs afterwards
# (either a duplicate event or death), so these myocardial events can be removed, 
#i will do the same for stroke as well. this ensures only non fatal myocardial infarctions 
# and strokes are captured in the MACE outcomes. fatal myocardial infarctions and stroke would fall into CV death 
mace_data_filtered_mi_cvd <- mace_data %>%
  group_by(person_id) %>% 
filter(event_type == 'myocardial infarction' & !((date.event - lag(date.event)) >= 30 &
                                                 (date.event - lag(date.event)) > 0)) %>%
  ungroup() 

mace_data_filtered_mi_cvd

person_id,event_type,date.event,event_id
<int>,<chr>,<date>,<dbl>
879366,myocardial infarction,2019-06-10,151.0
12394871,myocardial infarction,2019-03-12,130.0
12755225,myocardial infarction,2019-07-06,158.5
12831720,myocardial infarction,2018-01-09,74.0
13580091,myocardial infarction,2018-05-07,90.0
13580202,myocardial infarction,2016-06-06,1.0


In [37]:
mace_data <- mace_data %>% arrange(person_id, desc(date.event))


# this identifies stroke events where another event occurs afterwards(either a duplicate event or death), so these myocardial events can be removed, i will do the same for stroke as well. this ensures only non fatal myocardial infarctions and strokes are captured in the MACE outcomes. fatal myocardial infarctions and stroke would fall into CV death 
mace_filtered_stroke_cvd <- mace_data %>%
  group_by(person_id) %>% filter(event_type == 'stroke' & !((date.event - lag(date.event)) >= 30 & (date.event - lag(date.event)) > 0)) %>%
  ungroup() 

mace_filtered_stroke_cvd

person_id,event_type,date.event,event_id
<int>,<chr>,<date>,<dbl>
12921291,stroke,2016-12-29,10
13066369,stroke,2019-09-03,165
13424129,stroke,2019-05-13,146
13494181,stroke,2017-11-17,66
13723242,stroke,2017-02-25,20


In [38]:
#i then subsetted the mace_Data to remove those duplicate stroke and myocardial events identified above

In [39]:
#creat he not in function

'%notin%' <- Negate(`%in%`)

In [40]:
mace_data_filtered <- mace_data[mace_data$event_id %notin% mace_data_filtered_mi_cvd$event_id, ]

In [41]:
mace_data_filtered <- mace_data_filtered[mace_data_filtered$date.event %notin% mace_filtered_stroke_cvd$date.event,]

In [53]:
mace_data_filtered <- mace_data_filtered %>% arrange(person_id, desc(date.event))

table(mace_data_filtered$event_type)



             cv death myocardial infarction                stroke 
                  110                    17                    55 

In [43]:
# merge the care home cohort table in order to calculate the incidence density
#(need to subset by the cae_home_cohort_v1 table as cohort was updated in feb 2025, to exclude people with no follow-up 

In [44]:
sql1 <- 
paste(
    'SELECT person_id,first_episodestartdate, hypertension
from ',targetdb3,'.cb_2172_hypertension_combined 
where person_id in(select person_id from ',targetdb3,'.care_home_cohort_v1)
;',sep="")

bp_tbl <-bq_project_query(ProjectId,sql1)
ch_cohort <- bq_table_download(bp_tbl)

ch_cohort$person_id <- as.character(ch_cohort$person_id)
print(ch_cohort)

# A tibble: 2,774 × 3
   person_id first_episodestartdate hypertension
   <chr>     <dttm>                 <lgl>       
 1 13391566  2016-04-08 00:00:00    FALSE       
 2 13407837  2016-05-16 00:00:00    FALSE       
 3 13283147  2016-06-24 00:00:00    FALSE       
 4 12807410  2016-08-08 00:00:00    FALSE       
 5 13600637  2016-08-08 00:00:00    FALSE       
 6 13343064  2016-08-08 00:00:00    FALSE       
 7 13256011  2016-08-16 00:00:00    FALSE       
 8 12515587  2016-09-16 00:00:00    FALSE       
 9 13420182  2016-09-16 00:00:00    FALSE       
10 13355901  2016-10-24 00:00:00    FALSE       
# ℹ 2,764 more rows


In [45]:
ch_mace_merged <- merge(x =ch_cohort, y = mace_data_filtered, all.x = TRUE)

In [46]:
ch_mace_merged$first_episodestartdate <- as_date(ch_mace_merged$first_episodestartdate)

In [47]:
ch_mace_merged <- ch_mace_merged %>% mutate(exposure_time = (date.event -first_episodestartdate))

In [48]:
 ch_mace_merged_filtered <- ch_mace_merged %>% filter(hypertension == TRUE)

In [49]:
ch_mace_merged_filtered$exposure_time <- ifelse(is.na(ch_mace_merged_filtered$exposure_time) ,365,ch_mace_merged_filtered$exposure_time)

In [50]:
ch_mace_merged_filtered <-ch_mace_merged_filtered %>%  select(-"event_id")

In [51]:
(sum(ch_mace_merged_filtered$exposure_time < 365)/(ch_mace_merged_filtered$exposure_time %>% sum()/365))*100

[1] 7.498732